# Notebook 04 — My Report Runs Out of Memory
## Right-Sizing Warehouses and Accelerating Ad-Hoc Queries

**What you'll learn**: When a complex query "spills to disk," it means your warehouse is too small. Here's how to diagnose it, and when to ask for a bigger warehouse vs. an accelerated one.

| | Right Approach | Wrong Approach |
|---|---|---|
| **Sizing** | Match warehouse to query complexity | Use XS for everything (spills to remote storage) |
| **Ad-hoc analytics** | Enable Query Acceleration (QAS) | Wait for slow serial execution |
| **Cost** | Bigger warehouse finishes faster = cheaper overall | Small + slow = same cost, longer wait |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 1: Diagnosing the Problem — "My Fraud Report Takes Forever"

**Scenario**: You're building a monthly fraud loss report — it aggregates 500M rows by account type, channel, and month. On your default XS warehouse, it's painfully slow.

### Run the report on XS (watch it struggle)

In [ ]:
USE WAREHOUSE HOL_XS;

-- Complex aggregation on XS — will spill
SELECT
    a.account_type,
    t.channel,
    DATE_TRUNC('month', t.transaction_date) AS txn_month,
    COUNT(*) AS txn_count,
    SUM(t.amount) AS total_amount,
    SUM(CASE WHEN t.fraud_flag THEN t.amount ELSE 0 END) AS fraud_losses
FROM TRANSACTIONS t
JOIN ACCOUNTS a ON t.account_id = a.account_id
WHERE t.transaction_date >= '2024-01-01'
GROUP BY 1, 2, 3
ORDER BY fraud_losses DESC;

**Check the Query Profile** → Look for:
- `Bytes spilled to local storage` — the warehouse ran out of memory and is using disk (slow)
- `Bytes spilled to remote storage` — even disk wasn't enough, it's writing to cloud storage (very slow!)

This is like trying to do a year-end reconciliation on a laptop with 4GB RAM — it technically works, but takes 10x longer.

### The Fix: Run the same report on a Medium warehouse (fits in memory)

In [ ]:
USE WAREHOUSE HOL_M;

-- Same query on Medium — fits in memory
SELECT
    a.account_type,
    t.channel,
    DATE_TRUNC('month', t.transaction_date) AS txn_month,
    COUNT(*) AS txn_count,
    SUM(t.amount) AS total_amount,
    SUM(CASE WHEN t.fraud_flag THEN t.amount ELSE 0 END) AS fraud_losses
FROM TRANSACTIONS t
JOIN ACCOUNTS a ON t.account_id = a.account_id
WHERE t.transaction_date >= '2024-01-01'
GROUP BY 1, 2, 3
ORDER BY fraud_losses DESC;

**Check the Query Profile** → Compare:
- Zero spilling
- Significantly faster execution
- Credits used: 4x more per hour BUT query finishes 5-10x faster → net cheaper

---
## Step 2: Compare Metrics

In [ ]:
SELECT
    warehouse_name,
    warehouse_size,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_spilled_to_local_storage / (1024*1024*1024) AS spill_local_gb,
    bytes_spilled_to_remote_storage / (1024*1024*1024) AS spill_remote_gb,
    partitions_scanned
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -15, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
WHERE start_time > DATEADD(minute, -15, CURRENT_TIMESTAMP())
WHERE query_text ILIKE '%fraud_losses%'
ORDER BY start_time DESC
LIMIT 2;

---
## Step 3: Query Acceleration — Turbo Mode for Ad-Hoc Analysis

**Scenario**: You're a fraud analyst doing ad-hoc exploration — "Find all transactions over $500 in November grouped by merchant." QAS temporarily adds more compute power to speed up large scans.

### Without Query Acceleration

In [ ]:
-- Create a warehouse WITHOUT QAS
CREATE WAREHOUSE IF NOT EXISTS HOL_NO_QAS
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    ENABLE_QUERY_ACCELERATION = FALSE;

USE WAREHOUSE HOL_NO_QAS;

SELECT
    merchant_name,
    COUNT(*) AS high_value_txns,
    SUM(amount) AS total_amount
FROM TRANSACTIONS
WHERE amount > 500
  AND transaction_date >= '2024-11-01'
GROUP BY 1
ORDER BY total_amount DESC
LIMIT 20;

### With QAS Enabled

In [ ]:
-- Create a warehouse WITH QAS
CREATE WAREHOUSE IF NOT EXISTS HOL_QAS
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    ENABLE_QUERY_ACCELERATION = TRUE
    QUERY_ACCELERATION_MAX_SCALE_FACTOR = 8;

USE WAREHOUSE HOL_QAS;

SELECT
    merchant_name,
    COUNT(*) AS high_value_txns,
    SUM(amount) AS total_amount
FROM TRANSACTIONS
WHERE amount > 500
  AND transaction_date >= '2024-11-01'
GROUP BY 1
ORDER BY total_amount DESC
LIMIT 20;

**Check Query Profile** → Look for "Query Acceleration" section showing partitions offloaded to the service.

---
## Step 4: Identify QAS-Eligible Queries

In [ ]:
-- Which queries would benefit most from QAS?
SELECT
    query_id,
    SUBSTR(query_text, 1, 80) AS query_preview,
    eligible_query_acceleration_time / 1000 AS eligible_time_sec
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ACCELERATION_ELIGIBLE
WHERE start_time > DATEADD(hour, -1, CURRENT_TIMESTAMP())
ORDER BY eligible_query_acceleration_time DESC
LIMIT 5;

---
## Key Takeaways

| Symptom | Diagnosis | What to Ask For |
|---------|-----------|----------------|
| Query Profile shows "spilling" | Warehouse is too small for this query | *"Can I get a MEDIUM warehouse for my monthly reports?"* |
| Query is fast with zero spilling | Warehouse is right-sized | Don't change anything! |
| Ad-hoc fraud scans are slow | Need Query Acceleration | *"Can we enable QAS on the analytics warehouse?"* |
| Dashboard queries are queued | Too many users on one warehouse | *"Can we enable auto-scaling (multi-cluster)?"* |

**Rule of thumb**: If your query spills → ask for a bigger warehouse. If it doesn't spill on XS → don't upsize.

**💡 What to tell your platform team**:  
*"My monthly aggregation query spills 2 GB to remote storage on XS. Can I use a MEDIUM warehouse for monthly reporting? Here's the Query Profile."*

**Next** → [Notebook 05 — Finding a Needle in 500M Rows](./Notebook_05_Search_Optimization.ipynb)

---
## Cleanup

In [ ]:
USE WAREHOUSE HOL_XS;
-- DROP WAREHOUSE IF EXISTS HOL_NO_QAS;
-- DROP WAREHOUSE IF EXISTS HOL_QAS;